In [1]:
import numpy as np
import pandas as pd

In [2]:
catalyst_df = pd.read_csv("datasets/catalyst_ds.csv")
market_data_df = pd.DataFrame(pd.read_csv("datasets/market_ds.csv"))
final_df = pd.DataFrame(pd.read_csv("datasets/final_ds_NCAR20.csv"))

In [3]:
def trend(market_data_df, ticker, eventDate):
    df = (market_data_df[market_data_df['Ticker'] == ticker].sort_values('Date').reset_index(drop=True))
    match = df.index[pd.to_datetime(df['Date'], utc = True).dt.strftime('%Y-%m-%d') == pd.to_datetime(eventDate, utc = True).strftime('%Y-%m-%d')]

    if len(match)==0:
        return np.nan
    idx = match[0]

    if idx < 60:
        return None
     
    # extract stock values at specified timepoints
    s0, s30, s60 = df.loc[[idx, idx-30, idx-60], 'Close'].values
    n0, n30, n60 = df.loc[[idx, idx-30, idx-60], 'NBI'].values

    # calculate trends
    return {
        'Stock_Trend_30d': (s0-s30) / s30,
        'Stock_Trend_60d': (s0-s60) / s60,
        'NBI_Trend_30d': (n0-n30) / n30,
        'NBI_Trend_60d': (n0-n60) / n60,
    }

In [4]:
trends_res = [trend(market_data_df, ticker, date)
                          for ticker, date in zip(final_df['company_ticker'], 
                                                  final_df['date'])]
trends_df = pd.DataFrame([res if isinstance(res, dict) else {} for res in trends_res])
trends_df[['company_ticker', 'date']] = final_df[['company_ticker', 'date']].values

In [5]:
final_df = pd.merge(final_df, 
                    trends_df,
                    left_on=['company_ticker', 'date'],
                    right_on=['company_ticker', 'date'],
                    how = 'left',
                    )

In [6]:
final_df.head()

,catalyst,date,company_name,company_ticker,drug_name,nct_number,label,press_link,stage,polarity,...,Cost Of Revenue,Operating Revenue,Cash And Cash Equivalents,Total Debt,Capital Expenditure,drug_count,Stock_Trend_30d,Stock_Trend_60d,NBI_Trend_30d,NBI_Trend_60d
0,Phase 1 trial recently initiated noted July 17...,2025-07-17,Praxis Precision Medicines Inc.,PRAX,Relutrigine (PRAX-562) - (EMERALD),NCT07010471,Phase 1,https://www.biopharmcatalyst.com/company/PRAX/...,Phase 1,0.032329,...,NaN,0.0,357329000.0,110000.0,-56000.0,4,0.333495,0.554612,0.064976,0.130625
1,Phase 1 data published in the peer-reviewed jo...,2025-03-07,Cerenome Inc Com Par $0.0001 (New) Jan 2026,PSTV,REYOBIQ (rhenium Re186 obisbemeda) - (ReSPECT-...,NCT01906385,Phase 1,https://www.biopharmcatalyst.com/company/PSTV/...,Phase 1,0.902388,...,NaN,0.0,4256000.0,821000.0,-67000.0,3,NaN,NaN,NaN,NaN
2,Phase 1 trial of CUE-101 showed an additional ...,2025-07-16,Cue Biopharma Inc.,CUE,CUE-101,NCT03978689,Phase 1,https://www.biopharmcatalyst.com/company/CUE/n...,Phase 1,0.784513,...,NaN,27466000.0,27136000.0,4197000.0,-177000.0,2,0.310873,0.123360,0.073051,0.120741
3,Phase 1/1b data showed an overall response rat...,2025-02-14,Cellectis S.A.,CLLS,ALLO-501/501A + ALLO-647 - (ALPHA2),NCT04416984,Phase 1,https://www.biopharmcatalyst.com/company/ALLO/...,Phase 1,0.904840,...,9797000.0,72949000.0,61533000.0,90733000.0,-3533000.0,4,-0.138889,-0.171123,0.049000,0.027404
4,Phase 1/1b data showed an overall response rat...,2025-02-14,Allogene Therapeutics Inc.,ALLO,ALLO-501/501A + ALLO-647 - (ALPHA2),NCT04416984,Phase 1,https://www.biopharmcatalyst.com/company/ALLO/...,Phase 1,0.904840,...,NaN,0.0,51688000.0,83253000.0,-386000.0,3,-0.042254,-0.101322,0.049000,0.027404


In [7]:
final_df.to_csv("datasets/final_ds.csv", index = False)